# ALBERT — A Lite BERT for Self-supervised Learning of Language Representations (2019)

_Papers · Transformers & LLMs_

**Shrink BERT with two parameter-reduction tricks — factorize the embedding and share one block across all layers — and swap next-sentence prediction for sentence-order prediction.**

---

This notebook is a practice scaffold for the **ALBERT — A Lite BERT for Self-supervised Learning of Language Representations (2019)** lesson. We build it up one step at a time: count the parameters the two tricks save, assemble a tiny ALBERT-style encoder, and watch weight sharing trade a little accuracy for a big parameter cut. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We build a miniature ALBERT in four steps: (1) count the parameters the embedding factorization saves, (2) define a standard Transformer encoder block, (3) wire up a tiny ALBERT-style encoder whose two flags toggle the parameter-reduction tricks, and (4) train it on a small masked-language task to see what cross-layer sharing costs.

### Step 1 — Count what the embedding factorization saves

BERT stores one $H$-dimensional vector per vocabulary token, a $V\times H$ table. ALBERT instead keeps a smaller $V\times E$ table and a single shared $E\times H$ up-projection. Because $V$ is huge (30,000) and $E$ is much smaller than $H$, this slashes the embedding parameters. Here we plug in the paper's ALBERT-base numbers (Table 1) and confirm the ~6x cut.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# Worked example: ALBERT-base embedding factorization (Table 1: V=30000, H=768, E=128).
V_big, H_big, E_big = 30000, 768, 128

emb_bert = V_big * H_big                      # one H-dim vector per token: 23,040,000
emb_albert = V_big * E_big + E_big * H_big    # V->E table plus E->H up-projection: 3,938,304

saving = emb_bert - emb_albert               # absolute parameters removed
ratio = emb_bert / emb_albert                # how many times smaller

print("BERT-base embedding   V*H      =", emb_bert)        # 23040000
print("ALBERT-base embedding V*E+E*H  =", emb_albert,
      "=", V_big * E_big, "+", E_big * H_big)               # 3938304 = 3840000 + 98304
print("saving =", saving, " ratio = %.2fx smaller" % ratio)   # 19101696, 5.85x

### Step 2 — Build a standard Transformer encoder block

ALBERT reuses the same encoder block as BERT — multi-head self-attention followed by a feed-forward network, each wrapped in a residual connection and LayerNorm. We define it once here; the ALBERT tricks in the next step only change *how many distinct copies* of this block exist, not what the block does.

In [ ]:
# A standard Transformer encoder block (from paper-transformer): MHA + FFN, residual + LayerNorm.
class MHA(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h = h
        self.dk = d // h
        self.Wq, self.Wk, self.Wv, self.Wo = (nn.Linear(d, d) for _ in range(4))

    def _split(self, x):
        B, S, _ = x.shape
        return x.view(B, S, self.h, self.dk).transpose(1, 2)   # (B, heads, S, dk)

    def forward(self, x):
        Q = self._split(self.Wq(x))
        K = self._split(self.Wk(x))
        Vv = self._split(self.Wv(x))
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.dk)
        attn = F.softmax(scores, dim=-1) @ Vv
        B, _, S, _ = attn.shape
        merged = attn.transpose(1, 2).contiguous().view(B, S, self.h * self.dk)
        return self.Wo(merged)

class EncoderBlock(nn.Module):
    def __init__(self, d, h, ff):
        super().__init__()
        self.attn = MHA(d, h)
        self.ff = nn.Sequential(nn.Linear(d, ff), nn.ReLU(), nn.Linear(ff, d))
        self.n1 = nn.LayerNorm(d)
        self.n2 = nn.LayerNorm(d)

    def forward(self, x):
        x = self.n1(x + self.attn(x))   # attention sub-layer + residual + norm
        return self.n2(x + self.ff(x))  # feed-forward sub-layer + residual + norm

### Step 3 — Wire up a tiny ALBERT and compare parameter counts

`TinyALBERT` has two flags. `factorized` chooses the $V\to E\to H$ embedding (ALBERT) versus a direct $V\to H$ embedding (BERT). `shared` chooses *one* encoder block reused $L$ times (ALBERT) versus $L$ distinct blocks (BERT). With both tricks on, the model is just as deep but holds far fewer parameters — which we confirm by counting both variants.

In [ ]:
# Tiny ALBERT-style encoder. Two flags toggle the tricks (and the ablation).
V, H, E, L, h, ff = 32, 64, 16, 4, 4, 128   # tiny vocab/width so it trains fast
SEQ, B = 12, 64
MASK = 1                                    # token id reserved for [MASK]

def positional_encoding(n, d):
    pos = torch.arange(n).unsqueeze(1).float()
    i2 = torch.arange(0, d, 2).float()
    den = torch.pow(10000.0, i2 / d)
    pe = torch.zeros(n, d)
    pe[:, 0::2] = torch.sin(pos / den)
    pe[:, 1::2] = torch.cos(pos / den)
    return pe

class TinyALBERT(nn.Module):
    def __init__(self, factorized=True, shared=True):
        super().__init__()
        self.shared = shared
        if factorized:                                  # ALBERT: V->E then E->H  (\S 3.1)
            self.emb = nn.Embedding(V, E)
            self.proj = nn.Linear(E, H)
        else:                                           # BERT: V->H directly
            self.emb = nn.Embedding(V, H)
            self.proj = None
        self.register_buffer("pe", positional_encoding(SEQ, H))
        if shared:                                      # ALBERT: ONE block reused L times (\S 3.1)
            self.block = EncoderBlock(H, h, ff)
            self.blocks = None
        else:                                           # BERT: L distinct blocks (ablation)
            self.blocks = nn.ModuleList([EncoderBlock(H, h, ff) for _ in range(L)])
            self.block = None
        self.head = nn.Linear(H, V)                     # MLM head: predict the token at each position

    def forward(self, t):
        x = self.emb(t)
        if self.proj is not None:
            x = self.proj(x)                            # up-project E -> H
        x = x + self.pe[:x.shape[1]]
        if self.shared:
            for _ in range(L):                          # same weights, applied L times (still L deep)
                x = self.block(x)
        else:
            for b in self.blocks:
                x = b(x)
        return self.head(x)

    def nparams(self):
        return sum(p.numel() for p in self.parameters())

# Parameter-count comparison on the tiny model.
bert_style = TinyALBERT(factorized=False, shared=False)    # no tricks
albert_style = TinyALBERT(factorized=True,  shared=True)   # both tricks
print("\ntiny BERT-style   (no factorize, no share) params:", bert_style.nparams())
print("tiny ALBERT-style (factorize + share)      params:", albert_style.nparams())
print("tiny embedding  V*H =", V * H, "  vs  V*E+E*H =", V * E + E * H)

### Step 4 — Train an MLM and measure what sharing costs

We give the model a learnable masked-language task: each sequence is an arithmetic progression, so a masked token is recoverable from its neighbors. We train two variants — sharing ON and sharing OFF — with identical data, optimizer, and seed, and compare final masked-token accuracy against parameter count. This is ALBERT's bet: a small accuracy drop for a large parameter cut (§ 3.1, Table 4).

In [ ]:
# Small MLM task with LEARNABLE structure: arithmetic-progression sequences.
# token[i] = (start + i*step) mod (V-2) + 2  -> a masked token is recoverable from its neighbors.
def mask_batch():
    start = torch.randint(0, V - 2, (B, 1))
    step = torch.randint(1, 5, (B, 1))
    idx = torch.arange(SEQ).unsqueeze(0)
    t = ((start + idx * step) % (V - 2)) + 2          # tokens 2..V-1 (0,1 reserved)
    inp = t.clone()
    m = (torch.rand(B, SEQ) < 0.15)                   # mask ~15% (BERT/ALBERT rate)
    inp[m] = MASK
    return inp, t, m

def train(factorized, shared, steps=600, lr=3e-3):
    torch.manual_seed(0)
    net = TinyALBERT(factorized, shared)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    for s in range(steps):
        inp, tgt, m = mask_batch()
        logits = net(inp)
        loss = F.cross_entropy(logits[m], tgt[m])     # loss over masked positions only
        opt.zero_grad()
        loss.backward()
        opt.step()
        if s % 150 == 0 or s == steps - 1:
            acc = (logits[m].argmax(-1) == tgt[m]).float().mean().item()
            print(f"  step {s:4d}  loss {loss.item():.4f}  masked-acc {acc:.3f}")
    return net, acc

print("\nALBERT-style (factorize + SHARE):")
net_share, acc_share = train(factorized=True, shared=True)
print("BERT-style ablation (factorize, NO share):")
net_nosh, acc_nosh = train(factorized=True, shared=False)
print(f"\nfinal masked-acc  shared: {acc_share:.3f} ({net_share.nparams()} params)"
      f"   not-shared: {acc_nosh:.3f} ({net_nosh.nparams()} params)")
# shared reaches ~0.80 acc with ~37k params; not-shared ~0.87 with ~138k params (~3.7x more).
# Sharing trades a small accuracy drop for a big parameter cut -- ALBERT's bet (\S 3.1, Table 4).
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)

## Visualize the data & results

_Does ALBERT's cross-layer sharing (one block reused L times) keep MLM accuracy close to a BERT-style stack of L distinct blocks, while using far fewer parameters?_

### Step 1 — Re-declare the tiny model for a standalone run

This panel is self-contained so it runs even if you jump straight here. We re-import torch and redefine the same hyperparameters, positional-encoding table, attention, encoder block, and ALBERT module — identical to the reference implementation above, just gathered in one place.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

V, H, E, L, h, ff = 32, 64, 16, 4, 4, 128
SEQ, B, MASK = 12, 64, 1

def pe_table(n, d):
    pos = torch.arange(n).unsqueeze(1).float()
    i2 = torch.arange(0, d, 2).float()
    den = torch.pow(10000.0, i2 / d)
    t = torch.zeros(n, d)
    t[:, 0::2] = torch.sin(pos / den)
    t[:, 1::2] = torch.cos(pos / den)
    return t

class MHA(nn.Module):
    def __init__(s, d, h):
        super().__init__()
        s.h, s.dk = h, d // h
        s.Wq, s.Wk, s.Wv, s.Wo = (nn.Linear(d, d) for _ in range(4))

    def sp(s, x):
        B, S, _ = x.shape
        return x.view(B, S, s.h, s.dk).transpose(1, 2)

    def forward(s, x):
        Q, K, Vv = s.sp(s.Wq(x)), s.sp(s.Wk(x)), s.sp(s.Wv(x))
        a = F.softmax(Q @ K.transpose(-2, -1) / math.sqrt(s.dk), -1) @ Vv
        B, _, S, _ = a.shape
        return s.Wo(a.transpose(1, 2).contiguous().view(B, S, s.h * s.dk))

class Block(nn.Module):
    def __init__(s, d, h, ff):
        super().__init__()
        s.a = MHA(d, h)
        s.f = nn.Sequential(nn.Linear(d, ff), nn.ReLU(), nn.Linear(ff, d))
        s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)

    def forward(s, x):
        x = s.n1(x + s.a(x))
        return s.n2(x + s.f(x))

class ALBERT(nn.Module):
    def __init__(s, shared=True):
        super().__init__()
        s.shared = shared
        s.emb = nn.Embedding(V, E)
        s.proj = nn.Linear(E, H)   # factorized embedding (both runs)
        s.register_buffer("pe", pe_table(SEQ, H))
        if shared:
            s.block = Block(H, h, ff)
            s.blocks = None
        else:
            s.blocks = nn.ModuleList([Block(H, h, ff) for _ in range(L)])
            s.block = None
        s.head = nn.Linear(H, V)

    def forward(s, t):
        x = s.proj(s.emb(t)) + s.pe[:t.shape[1]]
        if s.shared:
            for _ in range(L):
                x = s.block(x)
        else:
            for b in s.blocks:
                x = b(x)
        return s.head(x)

    def n(s):
        return sum(p.numel() for p in s.parameters())

### Step 2 — Run both variants and record the accuracy curve

We train the shared and not-shared models again, this time saving the masked-token accuracy at every step so we can see the full learning curve rather than just the final number. Same data, optimizer, and seed for both, so any difference is due to weight sharing alone.

In [ ]:
def mask_batch():
    start = torch.randint(0, V - 2, (B, 1))
    step = torch.randint(1, 5, (B, 1))
    idx = torch.arange(SEQ).unsqueeze(0)
    t = ((start + idx * step) % (V - 2)) + 2
    inp = t.clone()
    m = (torch.rand(B, SEQ) < 0.15)
    inp[m] = MASK
    return inp, t, m

def run(shared, steps=600):
    torch.manual_seed(0)
    net = ALBERT(shared)
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    accs = []
    for s in range(steps):
        inp, tgt, m = mask_batch()
        lg = net(inp)
        loss = F.cross_entropy(lg[m], tgt[m])
        opt.zero_grad()
        loss.backward()
        opt.step()
        accs.append((lg[m].argmax(-1) == tgt[m]).float().mean().item())
    return net, accs

ns, sh = run(True)
nn_, no = run(False)

### Step 3 — Read off the trade-off

We sample the accuracy curves at a few checkpoints and print each variant's parameter count. The shared model lands near ~0.80 accuracy with ~37k parameters; the not-shared model reaches ~0.87 with ~138k parameters (~3.7x more) — a small accuracy cost for a big parameter saving.

In [ ]:
idx = list(range(0, 600, 50)) + [599]
print("shared  (%d params):" % ns.n(),  [[i, round(sh[i], 3)] for i in idx])
print("noshare (%d params):" % nn_.n(), [[i, round(no[i], 3)] for i in idx])
# shared -> ~0.80 acc, ~37k params; noshare -> ~0.87 acc, ~138k params (~3.7x more).

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The sharing ablation. Your tiny encoder trains an MLM. With cross-layer sharing ON
            (ALBERT-style, one block applied $L$ times) it reaches ~0.80 masked-token accuracy with about
            37k parameters; turn sharing OFF (BERT-style, $L$ distinct blocks) and it reaches ~0.87 with about
            138k parameters. What is the trade-off ALBERT is making, and does it match the paper's Table 4?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change only the layer wiring: ALBERT-style uses one EncoderBlock looped $L$ times; the ablation uses an nn.ModuleList of $L$ distinct blocks. Keep $E$, $H$, $L$, data, optimizer, and seed identical. — _An honest ablation changes exactly one thing &mdash; weight sharing &mdash; so any difference is attributable to it._
- Compare both axes: parameters (37k shared vs 138k not-shared, ~3.7&times; fewer) and final accuracy (~0.80 vs ~0.87). — _Sharing's value is parameters-per-accuracy: a small accuracy cost buys a large parameter cut._
- Relate to the paper: Table 4 reports sharing-all costs only a small drop in the downstream average versus not sharing (about &minus;1.5 absolute for ALBERT-base). — _The qualitative effect &mdash; big parameter cut, small accuracy cost &mdash; reproduces on toy data even though the absolute numbers are ours, not the paper's._

**Answer:** Sharing trades a small accuracy drop for a large parameter cut: in our run ~0.80 vs ~0.87
                 accuracy but ~3.7&times; fewer parameters (37k vs 138k). That is exactly ALBERT's bet &mdash;
                 the redundant per-layer weights buy little, so removing them costs little. It matches the
                 direction of Table 4, where sharing all parameters drops the downstream average only modestly
                 (~&minus;1.5 for ALBERT-base) while shrinking the model. The CODEVIZ panel shows the two
                 training curves and the parameter counts side by side.

</details>

**Problem 2.** In the worked example ($V=30{,}000$, $H=768$, $E=128$) the embedding shrank from $23.0$M to
            $3.9$M parameters. Where did almost all of that saving come from, and why does the extra
            $E\times H$ matrix not matter?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Split the factorized count: $V\times E = 30{,}000\times128 = 3{,}840{,}000$ and $E\times H = 128\times768 = 98{,}304$. — _The token table $V\times E$ dominates because $V$ is huge; the up-projection $E\times H$ does not depend on $V$._
- Compare the big terms: $V\times H = 23{,}040{,}000$ vs $V\times E = 3{,}840{,}000$, a $6\times$ cut, because $E$ is $6\times$ smaller than $H$. — _The whole saving is the per-token vector going from width $H=768$ to width $E=128$._
- Note $98{,}304$ is ~0.4% of $23$M, so adding it back is negligible. — _The up-projection is one small matrix shared by all tokens, not one-per-token._

**Answer:** Almost all the saving comes from shrinking the per-token vector from $H=768$ to $E=128$:
                 the $V\times E$ table ($3.84$M) is $6\times$ smaller than the $V\times H$ table ($23.0$M)
                 because $E$ is $6\times$ smaller than $H$. The added $E\times H = 98{,}304$ up-projection is
                 a single shared matrix &mdash; about 0.4% of the original &mdash; so it does not undo the win.
                 This is why the factorization helps most when the vocabulary $V$ is large.

</details>

**Problem 3.** SOP vs NSP (conceptual ablation). Suppose you pre-train one model with BERT's NSP and
            another with ALBERT's SOP, then test each on the SOP task (decide if two same-document segments are
            in order or swapped). The paper finds the NSP-trained model scores ~52% (about chance) while the
            SOP-trained one scores ~86.5%. Why can't the NSP-trained model do SOP?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall NSP's negative: sentence B is drawn from a different document, so topic usually differs from A. — _NSP can be solved by topic-matching alone &mdash; "are these about the same subject?" &mdash; without modeling order or coherence._
- Recall SOP's negative: the same two consecutive segments with their order swapped &mdash; identical topic, only the order flipped. — _Topic gives zero signal on SOP, so the model must judge discourse coherence/order._
- Conclude an NSP-trained model never learned order, so on SOP it is near chance (~52%); an SOP-trained model did, so it scores ~86.5%. — _Each loss teaches the skill its own negatives require; only SOP's negatives require coherence._

**Answer:** NSP's negative comes from a different document, so it is solvable by topic alone &mdash; the
                 model never has to learn sentence order. SOP's negative is the same two segments with order
                 swapped, so topic is useless and only coherence/order separates positive from negative. A model
                 trained on NSP therefore never acquired the ordering skill and scores ~52% (chance) on SOP,
                 while the SOP-trained model scores ~86.5%. This is why ALBERT replaced NSP with SOP for
                 multi-sentence downstream tasks (&sect;3.1, Table 5).

</details>